# Vidal LSTM Transfer Learning: Panasonic → CMU eVTOL

This notebook is the **Vidal method experiment entrypoint and audit surface**. It reuses repository modules for the model, CMU adapter, resampling, windowing, normalization, and metrics. It does not retrain Vidal Case 1–4 or invent an unfrozen CMU adaptation protocol.

# 0. Status and Experiment Manifest

- Paper: Vidal et al. (2019)
- Method: LSTM SOC estimator with supervised transfer learning
- Source domain: Panasonic NCR18650PF, 25 °C
- Target domain: CMU Sony-Murata VTC6 eVTOL
- Adaptation type: supervised
- Target true reference labels used: true only during future adaptation
- Pseudo labels used: false
- Current scope: source artifact audit, CMU input QC, and zero-shot evaluation

In [ ]:
from __future__ import annotations

import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

RUN_SOURCE_TRAINING = False
RUN_ZERO_SHOT_SMOKE_TEST = True
RUN_FULL_ZERO_SHOT = False
RUN_SUPERVISED_ADAPTATION = False
WRITE_RESULTS = False
SEED = 42

def find_repo_root(start: Path) -> Path:
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "pyproject.toml").is_file() and (path / "src/battery_benchmark").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside battery-soc-transfer-benchmark")

REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from battery_benchmark.datasets.cmu_evtol.vidal_adapter import (
    ADAPTER_VERSION,
    build_cmu_vidal_windows,
    load_cmu_dynamic_soc_for_vidal,
    transform_cmu_records,
)
from battery_benchmark.evaluation.metrics import compute_metrics
from battery_benchmark.preprocessing.normalization import MinMaxNormalizer
from battery_benchmark.reproductions.vidal_2019.model import VidalLSTM
from battery_benchmark.reproductions.vidal_2019.windows import FEATURES
from battery_benchmark.training.checkpointing import load_checkpoint

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
NOW_UTC = datetime.now(timezone.utc).isoformat()

LOCAL_ARTIFACT_ROOT = REPO_ROOT / "outputs/reproductions/vidal_2019"
EXTERNAL_ARTIFACT_ROOT = os.environ.get("VIDAL_ARTIFACT_ROOT")
ARTIFACT_ROOTS = [LOCAL_ARTIFACT_ROOT]
if EXTERNAL_ARTIFACT_ROOT:
    ARTIFACT_ROOTS.insert(0, Path(EXTERNAL_ARTIFACT_ROOT).expanduser())

CMU_BASE_ROOT = REPO_ROOT / "outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1"
CMU_EXTENSION_ROOT = REPO_ROOT / "outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1_extension4"
CMU_INVENTORY_PATH = REPO_ROOT / "outputs/datasets/cmu_evtol/vidal_adapter_v1/core7_inventory.csv"
RESULT_ROOT = REPO_ROOT / "outputs/benchmarks/panasonic_to_cmu_evtol/vidal"

manifest = {
    "method": "vidal_lstm_supervised_transfer",
    "paper": "Vidal et al. (2019)",
    "source_dataset": "panasonic_18650pf_25c",
    "target_dataset": "cmu_evtol_vtc6",
    "features": list(FEATURES),
    "target": "soc_dynamic_reference_not_ground_truth",
    "sampling_interval_s": 1.0,
    "sequence_length": 50,
    "stride": 50,
    "checkpoint_path": None,
    "scaler_path": None,
    "random_seed": SEED,
    "device": str(DEVICE),
    "data_version": "cmu_dynamic_capacity_soc_development_v1 + extension4",
    "git_commit": GIT_COMMIT,
    "implemented_sections": [0, 1, 2, 3, 4, 5, 7, 10, 11],
    "supervised_adaptation_status": "NOT_IMPLEMENTED",
}
environment = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "device": str(DEVICE),
    "cuda_available": torch.cuda.is_available(),
    "git_commit": GIT_COMMIT,
    "timestamp_utc": NOW_UTC,
    "repo_root": str(REPO_ROOT),
    "external_artifact_root_configured": bool(EXTERNAL_ARTIFACT_ROOT),
    "cmu_inventory_found": CMU_INVENTORY_PATH.is_file(),
}
display(pd.DataFrame(environment.items(), columns=["item", "value"]))
display(manifest)

# 1. Method Definition

The current reproduction uses voltage, current, and temperature as a 3-feature sequence input and predicts SOC at every time step. The architecture is one LSTM layer with hidden size 10 followed by a linear 10→1 head and an optional [0, 1] output clamp. The PyTorch recurrent-side secondary bias is fixed at zero so the model has the paper-aligned 571 trainable parameters.

Case 1 trains and evaluates within one battery dataset. Case 2 initializes from a completed Case 1 checkpoint and performs **supervised** target-domain fine-tuning using target reference SOC; it does not use pseudo labels. The current Case 2 implementation updates all 571 trainable parameters while the architecture-equivalence secondary bias remains frozen. This does not automatically freeze the future CMU protocol.

In [ ]:
model_config = yaml.safe_load((REPO_ROOT / "configs/reproductions/vidal_2019/model.yaml").read_text())
case1_config = yaml.safe_load((REPO_ROOT / "configs/reproductions/vidal_2019/vidal_case1_panasonic.yaml").read_text())
method_table = pd.DataFrame([
    ("feature order", list(FEATURES)),
    ("sequence length", case1_config.get("seq_len", "NOT_CONFIRMED")),
    ("hidden size", model_config.get("hidden_size", "NOT_CONFIRMED")),
    ("LSTM layers", model_config.get("num_layers", "NOT_CONFIRMED")),
    ("dropout", model_config.get("dropout", "NOT_CONFIRMED")),
    ("optimizer", "Adam (confirmed in Case 1 code)"),
    ("learning rate", case1_config.get("learning_rate", "NOT_CONFIRMED")),
    ("batch size", case1_config.get("batch_size", "NOT_CONFIRMED")),
    ("loss", "masked MSE (confirmed in Case 1 code)"),
    ("trainable parameters", VidalLSTM().count_trainable_parameters()),
], columns=["setting", "value"])
display(method_table)

# 2. Original-Paper Reproduction

This section reads existing Vidal Case 1/2 artifacts only. Paper-reported values that are not encoded in the repository remain `NOT_CONFIRMED`. Reproduction results are kept separate from CMU benchmark results.

In [ ]:
def discover_reproduction_metrics(roots: list[Path]) -> pd.DataFrame:
    rows = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob("metrics.json"):
            if path in seen:
                continue
            seen.add(path)
            try:
                metrics = json.loads(path.read_text())
                metadata_path = path.parent / "run_metadata.json"
                metadata = json.loads(metadata_path.read_text()) if metadata_path.is_file() else {}
                case = metadata.get("case", metrics.get("case", "NOT_CONFIRMED"))
                dataset = metadata.get("dataset", metadata.get("target_dataset", "NOT_CONFIRMED"))
                rows.append({
                    "case": case,
                    "dataset": dataset,
                    "mae": metrics.get("mae_fraction", metrics.get("mae", "NOT_AVAILABLE")),
                    "rmse": metrics.get("rmse_fraction", metrics.get("rmse", "NOT_AVAILABLE")),
                    "metrics_path": str(path),
                    "checkpoint": str(path.parent / "checkpoints/epoch_515.pt")
                        if (path.parent / "checkpoints/epoch_515.pt").is_file() else "NOT_AVAILABLE",
                })
            except (OSError, json.JSONDecodeError):
                continue
    return pd.DataFrame(rows, columns=["case", "dataset", "mae", "rmse", "metrics_path", "checkpoint"])

reproduction_results = discover_reproduction_metrics(ARTIFACT_ROOTS)
if reproduction_results.empty:
    display(pd.DataFrame([{
        "case": "Case 1/2", "paper_mae": "NOT_CONFIRMED", "paper_rmse": "NOT_CONFIRMED",
        "reproduction_mae": "NOT_AVAILABLE", "reproduction_rmse": "NOT_AVAILABLE",
        "difference": "NOT_AVAILABLE",
    }]))
else:
    display(reproduction_results)
print("RUN_SOURCE_TRAINING =", RUN_SOURCE_TRAINING, "— source retraining is outside this notebook.")

# 3. Source Checkpoint and Scaler Inventory

The canonical source pair must come from the **same completed Panasonic Case 1 run**: `checkpoints/epoch_515.pt` plus sibling `scaler.json`. Case 2 scalers are rejected because they were refit on a target dataset. Set `VIDAL_ARTIFACT_ROOT` before launching Jupyter when artifacts are mounted outside the repository.

In [ ]:
def find_panasonic_case1_pair(roots: list[Path]):
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        candidates.extend(root.rglob("checkpoints/epoch_515.pt"))
    for checkpoint in sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True):
        run_dir = checkpoint.parent.parent
        scaler = run_dir / "scaler.json"
        if not scaler.is_file():
            continue
        try:
            payload = torch.load(checkpoint, map_location="cpu", weights_only=False)
        except Exception:
            continue
        cfg = payload.get("config", {})
        dataset = cfg.get("dataset", payload.get("dataset"))
        if dataset == "panasonic_18650pf" and payload.get("epoch") == 514:
            return checkpoint, scaler
    return None, None

CHECKPOINT_PATH, SCALER_PATH = find_panasonic_case1_pair(ARTIFACT_ROOTS)
manifest["checkpoint_path"] = str(CHECKPOINT_PATH) if CHECKPOINT_PATH else None
manifest["scaler_path"] = str(SCALER_PATH) if SCALER_PATH else None

MODEL = None
SCALER = None
CHECKPOINT = None
source_inventory = {
    "checkpoint_status": "NOT_AVAILABLE",
    "scaler_status": "NOT_AVAILABLE",
    "same_run_directory": False,
    "model_loaded": False,
    "scaler_source_only_confirmed": False,
    "leakage_risk": "NOT_CONFIRMED",
}
if CHECKPOINT_PATH and SCALER_PATH:
    CHECKPOINT = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    scaler_json = json.loads(SCALER_PATH.read_text())
    MODEL = VidalLSTM().to(DEVICE)
    SCALER = MinMaxNormalizer()
    load_checkpoint(CHECKPOINT_PATH, MODEL, normalizer=SCALER, device=DEVICE)
    MODEL.eval()
    cfg = CHECKPOINT.get("config", {})
    source_inventory.update({
        "checkpoint_status": "PASS",
        "checkpoint_size_bytes": CHECKPOINT_PATH.stat().st_size,
        "checkpoint_mtime_utc": datetime.fromtimestamp(
            CHECKPOINT_PATH.stat().st_mtime, timezone.utc
        ).isoformat(),
        "checkpoint_keys": sorted(CHECKPOINT),
        "checkpoint_case": 1,
        "checkpoint_dataset": cfg.get("dataset"),
        "checkpoint_epoch_zero_based": CHECKPOINT.get("epoch"),
        "scaler_status": "PASS",
        "same_run_directory": CHECKPOINT_PATH.parent.parent == SCALER_PATH.parent,
        "model_loaded": True,
        "input_size": MODEL.input_size,
        "sequence_length": cfg.get("seq_len"),
        "scaler_features": scaler_json.get("feature_names", list(FEATURES)),
        "scaler_fit_split": scaler_json.get("fit_split", "NOT_CONFIRMED"),
        "scaler_feature_min": scaler_json.get("feature_min", scaler_json.get("min")),
        "scaler_feature_max": scaler_json.get("feature_max", scaler_json.get("max")),
        "scaler_source_only_confirmed": scaler_json.get("dataset") == "panasonic_18650pf"
            and scaler_json.get("fit_split") == "full_train",
        "leakage_risk": "PASS" if scaler_json.get("dataset") == "panasonic_18650pf"
            and scaler_json.get("fit_split") == "full_train" else "NOT_CONFIRMED",
    })
    with torch.no_grad():
        synthetic = torch.zeros((2, 50, 3), dtype=torch.float32, device=DEVICE)
        synthetic_prediction = MODEL(synthetic).cpu().numpy()
    source_inventory.update({
        "smoke_input_shape": list(synthetic.shape),
        "smoke_output_shape": list(synthetic_prediction.shape),
        "smoke_prediction_min": float(synthetic_prediction.min()),
        "smoke_prediction_max": float(synthetic_prediction.max()),
        "smoke_nonfinite_count": int((~np.isfinite(synthetic_prediction)).sum()),
    })

display(pd.DataFrame(source_inventory.items(), columns=["check", "value"]))
if MODEL is None:
    print("NOT_AVAILABLE: configure VIDAL_ARTIFACT_ROOT with the canonical Panasonic Case 1 run.")

# 4. CMU Adapter and Input QC

This section reads the committed core7 inventory and frozen dynamic-SOC parquet files. It uses the repository's CMU→Vidal adapter only for on-demand smoke/full evaluation; it does not regenerate adapter artifacts or fit a CMU scaler.

CMU SOC is a development reference label, not ground truth. The current adapter uses per-candidate 1 Hz resampling, no SOC clipping, and record-safe non-overlapping 50-step windows.

In [ ]:
CORE7 = {
    "VAH01": ("baseline", CMU_BASE_ROOT / "VAH01_dynamic_soc.parquet"),
    "VAH17": ("baseline", CMU_BASE_ROOT / "VAH17_dynamic_soc.parquet"),
    "VAH27": ("baseline", CMU_BASE_ROOT / "VAH27_dynamic_soc.parquet"),
    "VAH30": ("35_c", CMU_EXTENSION_ROOT / "VAH30_dynamic_soc.parquet"),
    "VAH15": ("cruise_1000_s", CMU_EXTENSION_ROOT / "VAH15_dynamic_soc.parquet"),
    "VAH12": ("cruise_400_s", CMU_EXTENSION_ROOT / "VAH12_dynamic_soc.parquet"),
    "VAH11": ("discharge_power_minus_20_percent", CMU_EXTENSION_ROOT / "VAH11_dynamic_soc.parquet"),
}
core_inventory = pd.read_csv(CMU_INVENTORY_PATH) if CMU_INVENTORY_PATH.is_file() else pd.DataFrame()
inventory_rows, feature_ranges = [], []
for cell_id, (condition, path) in CORE7.items():
    base = core_inventory.loc[core_inventory["cell_id"] == cell_id].iloc[0].to_dict()         if not core_inventory.empty and cell_id in set(core_inventory["cell_id"]) else {}
    row = {
        "cell_id": cell_id,
        "condition": condition,
        "number_of_missions": base.get("vidal_records", "NOT_AVAILABLE"),
        "number_of_windows": base.get("vidal_windows", "NOT_AVAILABLE"),
        "sampling_interval_s": 1.0,
        "sequence_length": 50,
        "feature_columns": list(FEATURES),
        "target_column": "soc_dynamic",
        "SOC_min": base.get("soc_min", "NOT_AVAILABLE"),
        "SOC_max": base.get("soc_max", "NOT_AVAILABLE"),
        "has_nan": bool(base.get("nonfinite_count", 1)) if base else "NOT_AVAILABLE",
        "has_inf": bool(base.get("nonfinite_count", 1)) if base else "NOT_AVAILABLE",
        "file_path": str(path.relative_to(REPO_ROOT)) if path.is_file() else "NOT_AVAILABLE",
    }
    inventory_rows.append(row)
    if path.is_file():
        frame = pd.read_parquet(path, columns=["voltage_v", "current_a", "Temperature__C", "soc_dynamic"])
        for source, feature in [("voltage_v", "voltage"), ("current_a", "current"), ("Temperature__C", "temperature")]:
            values = frame[source].to_numpy(dtype=float)
            feature_ranges.append({
                "cell_id": cell_id, "feature": feature, "min": float(values.min()),
                "max": float(values.max()), "mean": float(values.mean()), "std": float(values.std()),
                "nonfinite": int((~np.isfinite(values)).sum()),
            })
cmu_inventory = pd.DataFrame(inventory_rows)
cmu_feature_stats = pd.DataFrame(feature_ranges)
display(cmu_inventory)
display(cmu_feature_stats)

adapter_qc = {
    "adapter_version": ADAPTER_VERSION,
    "feature_order": list(FEATURES),
    "feature_order_matches_checkpoint": list(FEATURES) == ["voltage", "current", "temperature"],
    "window_shape_expected": [50, 3],
    "cross_mission_windows": 0 if not core_inventory.empty else "NOT_CONFIRMED",
    "duplicate_windows": 0 if not core_inventory.empty else "NOT_CONFIRMED",
    "soc_clipping_enabled": False,
    "soc_clipping_count": int(core_inventory["soc_clipping_count"].sum()) if not core_inventory.empty else "NOT_CONFIRMED",
    "nonfinite_count": int(core_inventory["nonfinite_count"].sum()) if not core_inventory.empty else "NOT_CONFIRMED",
}
display(pd.DataFrame(adapter_qc.items(), columns=["check", "value"]))

In [ ]:
# Minimal diagnostics: one VAH01 mission and raw/source-scaler range comparisons.
if CORE7["VAH01"][1].is_file():
    vah01 = pd.read_parquet(CORE7["VAH01"][1])
    mission_id = vah01["mission_cycle_id"].iloc[0]
    mission = vah01[vah01["mission_cycle_id"] == mission_id]
    fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
    for ax, column, label in zip(
        axes,
        ["voltage_v", "current_a", "Temperature__C", "soc_dynamic"],
        ["Voltage (V)", "Current (A)", "Temperature (°C)", "Reference SOC"],
    ):
        ax.plot(mission["elapsed_candidate_s"], mission[column], linewidth=0.9)
        ax.set_ylabel(label)
    axes[-1].set_xlabel("Mission elapsed time (s)")
    fig.suptitle(f"VAH01 representative development candidate: {mission_id}")
    fig.tight_layout()
    plt.show()

if SCALER is not None and not cmu_feature_stats.empty:
    scaler_state = SCALER.state_dict()
    source_ranges = pd.DataFrame({
        "feature": list(FEATURES),
        "source_min": scaler_state["min"],
        "source_max": scaler_state["max"],
    })
    comparison = cmu_feature_stats.groupby("feature", as_index=False).agg(
        cmu_min=("min", "min"), cmu_max=("max", "max")
    ).merge(source_ranges, on="feature")
    comparison["below_source_fraction_status"] = "computed during evaluation"
    display(comparison)
    transformed_rows = []
    for cell_id, (_, path) in CORE7.items():
        if not path.is_file():
            continue
        sample = pd.read_parquet(path, columns=["voltage_v", "current_a", "Temperature__C"]).iloc[::100]
        x = sample.rename(columns={
            "voltage_v": "voltage", "current_a": "current", "Temperature__C": "temperature"
        })[list(FEATURES)].to_numpy(np.float32)
        z = SCALER.transform(x)
        for index, feature in enumerate(FEATURES):
            transformed_rows.append({
                "cell_id": cell_id, "feature": feature, "min": float(z[:, index].min()),
                "max": float(z[:, index].max()), "mean": float(z[:, index].mean()),
                "std": float(z[:, index].std()),
                "outside_0_1_fraction": float(np.mean((z[:, index] < 0) | (z[:, index] > 1))),
            })
    transformed_stats = pd.DataFrame(transformed_rows)
    display(transformed_stats)
else:
    print("NOT_RUN: source scaler unavailable; transformed CMU distribution and out-of-range ratios were not computed.")

# 5. Zero-Shot Evaluation

Definition: canonical Panasonic Case 1 checkpoint + frozen Panasonic scaler → no CMU training or parameter update → direct CMU prediction. Target reference labels are used only for evaluation.

Smoke defaults to VAH01 and at most three development candidates. Full zero-shot is opt-in and covers all seven cells. Metrics are computed per mission, then macro-averaged per cell; pooled-window metrics are not used as the sole summary.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def evaluate_zero_shot_cell(cell_id: str, max_missions=None):
    if MODEL is None or SCALER is None:
        raise RuntimeError("Canonical Panasonic Case 1 checkpoint/scaler pair is unavailable")
    path = CORE7[cell_id][1]
    if not path.is_file():
        raise FileNotFoundError(path)
    frame = pd.read_parquet(path)
    if max_missions is not None:
        keep = frame["mission_cycle_id"].drop_duplicates().iloc[:max_missions]
        frame = frame[frame["mission_cycle_id"].isin(keep)].copy()
    adapted = load_cmu_dynamic_soc_for_vidal([path]) if max_missions is None else None
    if adapted is None:
        from battery_benchmark.datasets.cmu_evtol.vidal_adapter import adapt_cmu_dynamic_soc
        adapted = adapt_cmu_dynamic_soc(frame, source_path=str(path), target_hz=1.0)
    records = transform_cmu_records(adapted.records, SCALER)
    windows = build_cmu_vidal_windows(records, seq_len=50, stride=50, keep_incomplete=False)
    dataset = TensorDataset(
        torch.from_numpy(windows.X), torch.from_numpy(windows.y), torch.arange(len(windows.X))
    )
    rows = []
    MODEL.eval()
    with torch.no_grad():
        for x, y, indices in DataLoader(dataset, batch_size=512, shuffle=False):
            prediction = MODEL(x.to(DEVICE)).cpu().numpy()
            for batch_index, window_index in enumerate(indices.tolist()):
                meta = windows.metadata[window_index]
                valid = int(meta["valid_length"])
                for step in range(valid):
                    reference = float(y[batch_index, step])
                    predicted = float(prediction[batch_index, step])
                    rows.append({
                        "model": "VidalLSTM", "paper": "Vidal et al. (2019)",
                        "track": "zero_shot", "adaptation_type": "none",
                        "fold": "source_only", "seed": SEED,
                        "source_dataset": "panasonic_18650pf_25c",
                        "adaptation_cells": "", "test_cell": cell_id,
                        "condition": CORE7[cell_id][0],
                        "target_labels_used": False, "pseudo_labels_used": False,
                        "split": "test", "phase": "NOT_AVAILABLE",
                        "mission_id": meta["mission_cycle_id"],
                        "window_index": window_index, "step_in_window": step,
                        "soc_reference": reference, "soc_prediction": predicted,
                        "absolute_error": abs(predicted - reference),
                    })
    predictions = pd.DataFrame(rows)
    metric_rows = []
    for mission_id, group in predictions.groupby("mission_id", sort=False):
        metric = compute_metrics(group["soc_reference"], group["soc_prediction"])
        metric_rows.append({
            "model": "VidalLSTM", "paper": "Vidal et al. (2019)",
            "track": "zero_shot", "adaptation_type": "none", "fold": "source_only",
            "seed": SEED, "source_dataset": "panasonic_18650pf_25c",
            "adaptation_cells": "", "test_cell": cell_id, "condition": CORE7[cell_id][0],
            "target_labels_used": False, "pseudo_labels_used": False,
            "split": "test", "phase": "all", "mission_id": mission_id,
            "n_windows": int(group["window_index"].nunique()),
            "mae": metric["mae"], "rmse": metric["rmse"],
            "checkpoint_path": str(CHECKPOINT_PATH), "scaler_path": str(SCALER_PATH),
            "config_path": str(REPO_ROOT / "configs/reproductions/vidal_2019/vidal_case1_panasonic.yaml"),
            "data_version": manifest["data_version"], "git_commit": GIT_COMMIT,
            "timestamp": NOW_UTC,
        })
    return predictions, pd.DataFrame(metric_rows)

zero_shot_predictions = pd.DataFrame()
zero_shot_mission_metrics = pd.DataFrame()
zero_shot_cell_metrics = pd.DataFrame()
ZERO_SHOT_SMOKE_STATUS = "NOT_RUN"
FULL_ZERO_SHOT_STATUS = "NOT_RUN"

if RUN_ZERO_SHOT_SMOKE_TEST:
    if MODEL is None or SCALER is None:
        print("NOT_RUN: canonical source checkpoint/scaler pair is unavailable.")
    else:
        zero_shot_predictions, zero_shot_mission_metrics = evaluate_zero_shot_cell("VAH01", max_missions=3)
        ZERO_SHOT_SMOKE_STATUS = "PASS"
        display(zero_shot_mission_metrics)
        print({
            "input_unit": "SOC fraction 0–1",
            "prediction_min": float(zero_shot_predictions["soc_prediction"].min()),
            "prediction_max": float(zero_shot_predictions["soc_prediction"].max()),
            "reference_min": float(zero_shot_predictions["soc_reference"].min()),
            "reference_max": float(zero_shot_predictions["soc_reference"].max()),
            "nonfinite_count": int((~np.isfinite(zero_shot_predictions[["soc_prediction", "soc_reference"]])).sum().sum()),
            "ordered_rows": len(zero_shot_predictions),
            "track": "zero_shot", "target_labels_used": False, "pseudo_labels_used": False,
        })

if RUN_FULL_ZERO_SHOT:
    if MODEL is None or SCALER is None:
        print("NOT_RUN: full zero-shot requires the canonical source artifact pair.")
    else:
        predictions, metrics = [], []
        for cell_id in CORE7:
            cell_predictions, cell_metrics = evaluate_zero_shot_cell(cell_id)
            predictions.append(cell_predictions)
            metrics.append(cell_metrics)
        zero_shot_predictions = pd.concat(predictions, ignore_index=True)
        zero_shot_mission_metrics = pd.concat(metrics, ignore_index=True)
        FULL_ZERO_SHOT_STATUS = "PASS"

if not zero_shot_mission_metrics.empty:
    zero_shot_cell_metrics = zero_shot_mission_metrics.groupby(
        ["model", "paper", "track", "test_cell", "condition"], as_index=False
    ).agg(
        n_missions=("mission_id", "nunique"),
        mission_macro_mae=("mae", "mean"),
        mission_macro_rmse=("rmse", "mean"),
    )
    display(zero_shot_cell_metrics)

In [ ]:
if not zero_shot_predictions.empty:
    example_mission = zero_shot_predictions["mission_id"].iloc[0]
    plot_frame = zero_shot_predictions[zero_shot_predictions["mission_id"] == example_mission]
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(plot_frame["soc_reference"].to_numpy(), label="reference")
    axes[0].plot(plot_frame["soc_prediction"].to_numpy(), label="prediction")
    axes[0].legend()
    axes[0].set_ylabel("SOC fraction")
    axes[0].set_title("track=zero_shot | target_labels_used=false | pseudo_labels_used=false")
    axes[1].plot(plot_frame["absolute_error"].to_numpy(), color="tab:red")
    axes[1].set_ylabel("Absolute error")
    axes[1].set_xlabel("Ordered predicted sample")
    fig.tight_layout()
    plt.show()

# 6. Vidal Original Transfer Protocol

**STATUS: NOT_IMPLEMENTED for CMU.**

The existing Case 2 reproduction is supervised: it loads only a completed Case 1 model state, creates a new optimizer and scheduler, refits a target scaler, and updates all 571 trainable parameters; the architecture-equivalence secondary LSTM bias remains fixed at zero. This notebook does **not** transfer that target-scaler behavior to CMU.

The following CMU choices remain `NOT_FROZEN`: target label quantity, train/validation ratio, layer-freezing policy, and early stopping. No training runs from this section.

In [ ]:
transfer_protocol = {
    "status": "NOT_IMPLEMENTED",
    "target_label_quantity": "NOT_FROZEN",
    "train_validation_ratio": "NOT_FROZEN",
    "layers_frozen": "NOT_FROZEN",
    "early_stopping": "NOT_FROZEN",
    "target_true_reference_labels_required": True,
    "pseudo_labels_used": False,
}
if RUN_SUPERVISED_ADAPTATION:
    raise RuntimeError("BLOCKED_BY_PROTOCOL: CMU supervised adaptation protocol is not frozen")
display(transfer_protocol)

# 7. CMU Baseline Three-Fold Supervised Adaptation

Only fold manifests and isolation checks are defined. Held-out cells may not participate in training, validation, scaler fitting, or hyperparameter selection. The Panasonic scaler stays frozen unless a later reviewed protocol explicitly changes that rule.

In [ ]:
folds = pd.DataFrame([
    {"fold": 1, "adaptation_cells": ["VAH17", "VAH27"], "held_out_test_cell": "VAH01"},
    {"fold": 2, "adaptation_cells": ["VAH01", "VAH27"], "held_out_test_cell": "VAH17"},
    {"fold": 3, "adaptation_cells": ["VAH01", "VAH17"], "held_out_test_cell": "VAH27"},
])
folds["isolation_check"] = folds.apply(
    lambda row: row["held_out_test_cell"] not in row["adaptation_cells"], axis=1
)
folds["status"] = "BLOCKED_BY_PROTOCOL"
display(folds)

# 8. Condition-Cell Stress Tests

**STATUS: NOT_IMPLEMENTED.** Future fold-specific adapted models will each evaluate VAH30, VAH15, VAH12, and VAH11, followed by mean ± standard deviation across the three models. Random models and zero-shot outputs are not substituted for missing adapted models.

# 9. Phase-Level Diagnostics

The requested evaluation phases are take-off, cruise, and landing, using the phase at the window's final target time point. Current CMU adapter outputs do not expose a reviewed formal phase label with those names. Therefore phase diagnostics are `NOT_CONFIRMED/NOT_RUN`; the notebook does not silently map development screening segments to formal phases.

# 10. Standardized Result Export

The repository output namespace is used: `outputs/benchmarks/panasonic_to_cmu_evtol/vidal/`. Writes are disabled unless `WRITE_RESULTS=True`. Smoke and full results are separated.

In [ ]:
export_status = "NOT_RUN"
if WRITE_RESULTS:
    if zero_shot_predictions.empty:
        print("NOT_RUN: no real zero-shot result exists to export.")
    else:
        scope = "full" if RUN_FULL_ZERO_SHOT and FULL_ZERO_SHOT_STATUS == "PASS" else "_smoke"
        output_dir = RESULT_ROOT / scope
        output_dir.mkdir(parents=True, exist_ok=True)
        zero_shot_predictions.to_parquet(output_dir / "zero_shot_predictions.parquet", index=False)
        zero_shot_mission_metrics.to_csv(output_dir / "zero_shot_mission_metrics.csv", index=False)
        zero_shot_cell_metrics.to_csv(output_dir / "zero_shot_cell_metrics.csv", index=False)
        (output_dir / "experiment_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
        qc = {
            "track": "zero_shot",
            "target_labels_used": False,
            "pseudo_labels_used": False,
            "smoke_status": ZERO_SHOT_SMOKE_STATUS,
            "full_status": FULL_ZERO_SHOT_STATUS,
            "n_prediction_rows": len(zero_shot_predictions),
            "n_missions": int(zero_shot_predictions["mission_id"].nunique()),
        }
        (output_dir / "qc_summary.json").write_text(json.dumps(qc, indent=2) + "\n")
        export_status = "PASS"
        print("Exported:", output_dir)
else:
    print("NOT_RUN: WRITE_RESULTS=False")

# 11. Final Validation Summary

In [ ]:
def status(condition, false_status: str = "FAIL") -> str:
    if condition is None:
        return "NOT_RUN"
    return "PASS" if condition else false_status

validation_summary = pd.DataFrame([
    ("Source checkpoint found", status(CHECKPOINT_PATH is not None, "NOT_RUN")),
    ("Source scaler found", status(SCALER_PATH is not None, "NOT_RUN")),
    ("Model loaded successfully", status(MODEL is not None, "NOT_RUN")),
    ("Feature order matched", status(tuple(FEATURES) == ("voltage", "current", "temperature"))),
    ("Sequence length matched", status(case1_config.get("seq_len") == 50)),
    ("CMU adapter files found", status(all(path.is_file() for _, path in CORE7.values()))),
    ("No cross-mission windows", status(adapter_qc["cross_mission_windows"] == 0)),
    ("No NaN/Inf", status(adapter_qc["nonfinite_count"] == 0)),
    ("Zero-shot smoke test passed", ZERO_SHOT_SMOKE_STATUS),
    ("Full zero-shot completed", FULL_ZERO_SHOT_STATUS),
    ("Supervised adaptation implemented", "NOT_IMPLEMENTED"),
    ("Condition stress test completed", "NOT_IMPLEMENTED"),
    ("Standardized export completed", export_status),
], columns=["check", "status"])
allowed = {"PASS", "FAIL", "NOT_RUN", "NOT_IMPLEMENTED", "NOT_CONFIRMED"}
assert set(validation_summary["status"]).issubset(allowed)
display(validation_summary)

print("Scientific boundary: CMU soc_dynamic is a development reference label, not ground truth.")
print("No source training, CMU scaler fitting, supervised adaptation, or pseudo-labeling was performed.")